In [ ]:
import numpy as np
import tifffile as tiff
from skimage.measure import label as label_connected_components
import os

Loading masks from: D:\Thesis_images\All_images_tif\annotations\proceesed_images_label\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label.tif
Volume shape: 75 slices, 1101x1101 pixels
Slice 0 processed. Found 6 objects.
Processed slice 10/75...
Processed slice 20/75...
Processed slice 30/75...
Processed slice 40/75...
Processed slice 50/75...
Processed slice 60/75...
Processed slice 70/75...
Saving tracked masks to: D:\Thesis_images\All_images_tif\annotations\proceesed_images_label\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_fixed.tif
Done!


In [3]:
import numpy as np
import tifffile
from scipy.ndimage import label as ndimage_label, generate_binary_structure

# Load your 3D label TIFF (shape: Z, Y, X)
labels_3d = tifffile.imread(r"D:\Thesis_images\All_images_tif\annotations\proceesed_images_label\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_vit_l_lm_patch1024\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_vit_l_lm_patch1024.tif").astype(np.uint32)

print(f"Original unique labels: {len(np.unique(labels_3d)) - 1}")  # -1 for background

# Create binary mask (object vs background)
binary_mask = labels_3d > 0

# Use 3D connected component labeling with 26-connectivity
# This assigns the SAME ID to all connected voxels, even if they had different IDs before
structure_3d = generate_binary_structure(3, 3)  # 26-connectivity (full 3D neighborhood)
result_labels, num_features = ndimage_label(binary_mask, structure=structure_3d)

print(f"After 3D connected component labeling: {num_features} connected objects")

# Save the result
output_path = r"D:\Thesis_images\All_images_tif\annotations\proceesed_images_label\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_vit_l_lm_patch1024\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_vit_l_lm_patch1024_synced.tif"
tifffile.imwrite(output_path, result_labels.astype(np.uint16))
print(f"Saved to {output_path}")


Original unique labels: 9
After 3D connected component labeling: 27 connected objects
Saved to D:\Thesis_images\All_images_tif\annotations\proceesed_images_label\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_vit_l_lm_patch1024\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_vit_l_lm_patch1024_synced.tif


In [4]:
import numpy as np
import tifffile
import os
from skimage.measure import regionprops
from skimage.filters import threshold_otsu
from skimage.morphology import remove_small_objects

# ==========================================
# 1. SETUP PATHS
# ==========================================
# This is the file you generated in the previous step
input_path = r"D:\Thesis_images\All_images_tif\annotations\proceesed_images_label\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_vit_l_lm_patch1024\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_vit_l_lm_patch1024_synced.tif"

# Automatically generate output path in the same folder
folder_path = os.path.dirname(input_path)
file_name = os.path.basename(input_path)
output_name = file_name.replace(".tif", "_cleaned_auto.tif")
output_path = os.path.join(folder_path, output_name)

print(f"Processing file: {file_name}")
print(f"Folder: {folder_path}")

# ==========================================
# 2. LOAD SYNCED LABELS
# ==========================================
print("Loading 3D image...")
labels_3d = tifffile.imread(input_path)

# ==========================================
# 3. AUTO-CALCULATE NOISE THRESHOLD
# ==========================================
print("Analyzing object sizes...")

# Get the size (volume) of every object
# Note: regionprops is usually fast enough for this on CPU
props = regionprops(labels_3d)
volumes = np.array([p.area for p in props])

if len(volumes) > 0:
    # Use Otsu's method on the LOG of sizes. 
    # Why Log? Biological cells vary wildly in size (100 vs 10,000). 
    # Log-scale makes the gap between "tiny noise" and "real cells" much clearer.
    log_volumes = np.log10(volumes)
    
    try:
        log_threshold = threshold_otsu(log_volumes)
        size_threshold = 10**log_threshold
    except Exception:
        # Fallback if statistical method fails (e.g. only 1 object type exists)
        size_threshold = np.percentile(volumes, 10) # Remove bottom 10% as safety

    print(f"   > Detected {len(volumes)} objects.")
    print(f"   > Auto-calculated Size Cutoff: {size_threshold:.2f} voxels")
else:
    print("   > Image is empty.")
    size_threshold = 0

# ==========================================
# 4. REMOVE SMALL BLOBS
# ==========================================
print(f"Removing objects smaller than {size_threshold:.2f} voxels...")

# Apply the filter
# min_size expects an integer. We cast the threshold to int.
clean_labels = remove_small_objects(labels_3d, min_size=int(size_threshold), connectivity=1)

final_count = len(np.unique(clean_labels)) - 1 # -1 for background
removed_count = len(volumes) - final_count

print(f"   > Removed {removed_count} small blobs.")
print(f"   > Remaining objects: {final_count}")

# ==========================================
# 5. SAVE RESULT
# ==========================================
print(f"Saving result to: {output_path}")

# Ensure we use a data type that fits the number of labels
if final_count > 65535:
    dtype = np.uint32
else:
    dtype = np.uint16

tifffile.imwrite(output_path, clean_labels.astype(dtype))
print("Done!")

Processing file: PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_vit_l_lm_patch1024_synced.tif
Folder: D:\Thesis_images\All_images_tif\annotations\proceesed_images_label\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_vit_l_lm_patch1024
Loading 3D image...
Analyzing object sizes...
   > Detected 27 objects.
   > Auto-calculated Size Cutoff: 981.74 voxels
Removing objects smaller than 981.74 voxels...
   > Removed 11 small blobs.
   > Remaining objects: 16
Saving result to: D:\Thesis_images\All_images_tif\annotations\proceesed_images_label\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_vit_l_lm_patch1024\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_vit_l_lm_patch1024_synced_cleaned_auto.tif
Done!


In [5]:
import numpy as np
import tifffile
import os
from skimage.morphology import remove_small_objects

# ==========================================
# 1. SETTINGS
# ==========================================
# Your input file
input_path = r"D:\Thesis_images\All_images_tif\annotations\proceesed_images_label\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_vit_l_lm_patch1024\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_vit_l_lm_patch1024_synced.tif"

# THE MAGIC NUMBER you found
# This removes everything <= 1449.
THRESHOLD = 1450 

# ==========================================
# 2. LOAD
# ==========================================
print(f"Loading image...")
labels_3d = tifffile.imread(input_path)

print(f"Original object count: {len(np.unique(labels_3d)) - 1}")

# ==========================================
# 3. APPLY FILTER
# ==========================================
print(f"Removing all objects smaller than {THRESHOLD} voxels...")

# min_size means "keep everything this size or bigger"
# So min_size=1450 will delete the 1449 blob.
clean_labels = remove_small_objects(labels_3d, min_size=THRESHOLD, connectivity=1)

new_count = len(np.unique(clean_labels)) - 1
print(f"New object count: {new_count}")
print(f"Removed {len(np.unique(labels_3d)) - 1 - new_count} blobs.")

# ==========================================
# 4. SAVE
# ==========================================
folder_path = os.path.dirname(input_path)
file_name = os.path.basename(input_path)
output_name = file_name.replace(".tif", f"_cleaned_{THRESHOLD}.tif")
output_path = os.path.join(folder_path, output_name)

print(f"Saving to: {output_path}")

# Check if we need 32-bit (if labels > 65k)
dtype = np.uint32 if new_count > 65535 else np.uint16
tifffile.imwrite(output_path, clean_labels.astype(dtype))

print("Done!")

Loading image...
Original object count: 27
Removing all objects smaller than 1450 voxels...
New object count: 16
Removed 11 blobs.
Saving to: D:\Thesis_images\All_images_tif\annotations\proceesed_images_label\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_vit_l_lm_patch1024\PN_KC_488_568_Brain1_CalyxL_LSMPlus_16bit_label_vit_l_lm_patch1024_synced_cleaned_1450.tif
Done!
